# Athena Advisor: prepare the data

Builds the StackSample training splits and converts the four public benchmarks (MixATIS, MixSNIPS, CLINC150,
NLBSE'22), then packs everything into `/kaggle/working/athena_data.zip`.

All logic lives in the `athena_advisor` package; this notebook only runs its scripts.

**Before you run** (notebook settings, right-hand panel):
1. **Internet:** on (needed to clone the repository and download the benchmarks).
2. **Input:** add the dataset `stackoverflow/stacksample`.
3. **Accelerator:** not needed for this notebook; a GPU is only used for training.
4. **Secrets (only if the repository is private):** add a secret named `GITHUB_TOKEN` with read access to the repository.

## 1. Environment

In [ ]:
!python --version
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU attached (fine for data preparation)"

## 2. Get the code

Tries a public clone first. If that fails, it uses the `GITHUB_TOKEN` secret, then removes the token from the
saved remote so it never appears in logs or outputs.

In [ ]:
import os
import subprocess

REPO = "https://github.com/pxsha23/ProjectAthena.git"
DEST = "/kaggle/working/ProjectAthena"
ML = f"{DEST}/ml"
# Never wait for a username prompt: fail fast instead.
os.environ["GIT_TERMINAL_PROMPT"] = "0"


def clone(url: str) -> bool:
    # Output is captured, not printed, so a URL that carries the token is never shown.
    return subprocess.run(["git", "clone", "--depth", "1", url, DEST], capture_output=True).returncode == 0


if not os.path.isdir(DEST):
    if not clone(REPO):
        from kaggle_secrets import UserSecretsClient

        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        if not clone(REPO.replace("https://", f"https://x-access-token:{token}@")):
            raise RuntimeError("Clone failed: check the GITHUB_TOKEN secret and that Internet is on.")
        subprocess.run(["git", "-C", DEST, "remote", "set-url", "origin", REPO], check=True)
        del token

!git -C {DEST} log -1 --format="Commit %h, %cd: %s"

## 3. Install

The package is installed without its dependencies so Kaggle's GPU build of PyTorch stays in place. Then only the
packages Kaggle does not ship are added, and the environment check confirms that everything imports.

In [ ]:
%pip install -q -e {ML} --no-deps
%pip install -q setfit optimum onnx onnxruntime

In [ ]:
os.environ["ATHENA_ML_ROOT"] = ML  # relative paths in the configs resolve to the ml/ folder
os.chdir(ML)
!python scripts/check_environment.py

## 4. Build the StackSample splits

Finds the attached StackSample files under `/kaggle/input`, labels every question from its tags, and writes the
chronological train / val_tune / val_cal / test splits with a data card.

In [ ]:
!python scripts/build_stacksample.py --input-root /kaggle/input --out data/processed/stacksample

## 5. Download and convert the benchmarks

Downloads the official files (pinned versions) and converts them to the common format, one data card each.

In [ ]:
!python scripts/download_benchmarks.py

## 6. Results

In [ ]:
!python scripts/summarize_data.py data/processed

In [ ]:
!cat data/processed/stacksample/DATA_CARD.md
!for card in data/processed/benchmarks/*/DATA_CARD.md; do echo; echo "==== $card"; cat "$card"; done

## 7. Package the output

Download `athena_data.zip` from the Output panel, or attach this notebook's output as the input of the training
notebook. The zip holds `data/processed/` only (no raw downloads); unzip it inside `ml/` to restore the same layout.

In [ ]:
import shutil

# base_dir keeps the data/processed/ paths inside the zip.
shutil.make_archive("/kaggle/working/athena_data", "zip", root_dir=".", base_dir="data/processed")
!ls -lh /kaggle/working/athena_data.zip